# AI Performance Engineering - Nebius Academy

## Assignment 1 - Evaluation:

### Task 1 - Define rubics:

1. Criterion definitions:
#### Fluency
- Good: Reads smoothly and naturally; no awkward phrasing.
- Ok: Mostly clear with minor awkward phrasing.
- Bad: Choppy or confusing; hard to follow.
#### Grammar
- Good: No noticeable spelling/grammar/punctuation errors.
- Ok: 1–2 minor errors that do not affect understanding.
- Bad: Multiple errors that distract or cause confusion.
#### Tone
- Good: Friendly, credible sales voice; positive and persuasive.
- Ok: Generally appropriate but slightly too dry, casual, or hyped.
- Bad: Clearly inappropriate (negative, rude, sarcastic, robotic).
#### Length
- Good: 50–90 words.
- Ok: 40–49 or 91–110 words.
- Bad: <40 or >110 words.
#### Grounding
- Good: All factual claims match provided product info; only light, generic persuasion.
- Ok: One minor, plausible inference not explicitly stated and not contradictory.
- Bad: Invents or contradicts key facts (material, warranty, features).
#### Latency (average time per call, lenient)
- Good: average latency ≤ 5.0 s.
- Ok: 5.0 s < average latency ≤ 9.0 s.
- Bad: average latency > 9.0 s.
#### Cost (normalized per 1K tokens: input + output)
- Good: ≤ 0.0001 USD per 1K tokens.
- Ok: 0.0001–0.0005 USD per 1K tokens.
- Bad: > 0.0005 USD per 1K tokens.
---------------------------------------------------
2. Pass / fail definition:
A description passes if:
- ≥3 criteria rated good
- ≤2 criteria rated ok
- 0 criteria rated bad
---------------------------------------------------
Formula:
- Let G = # good, O = # ok, B = # bad
Pass: G ≥ 3 ∧ O ≤ 2 ∧ B = 0
---------------------------------------------------
Go/No-Go Rules (Auto-Fail)
- Grounding ≠ good
- Length = bad
- Tone = bad
Formula:
Grounding ≠ good ∨ Length = bad ∨ Tone = bad → Fail



### Task 2 - generate the description for every product:

In [17]:
import os
import time
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
import tiktoken
from pydantic import BaseModel, Field
from typing import Literal

In [ ]:
load_dotenv()

NEBIUS_BASE_URL = "https://api.tokenfactory.nebius.com/v1/"
client = OpenAI(base_url=NEBIUS_BASE_URL, api_key=os.getenv("NEBIUS_API_KEY"))

GEN_CONFIG = {
    "model": "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "temperature": 0.7,
    "top_p": None,
    "top_k": None,
    "max_tokens": 150
}

#### Build the system prompt for description generation

In [ ]:
def build_system_prompt():
    return """You write concise product descriptions for an online catalog.

Write a clear, natural-sounding description based only on the provided product information.

Rules:
- Use only the provided product details
- Be friendly, credible, and positive
- Do not invent features, benefits, dimensions, or use cases
- Focus on what the product is and its relevant attributes
- Keep the description between 50 and 90 words
- Output only the final description
- Make it compelling for a customer ready to buy

Write ONLY the description. No extra text."""



#### Calculate cost per 1K tokens

In [5]:
def calculate_cost(input_tokens, output_tokens):
    """Calculate total cost per call in USD (Meta-Llama-3.1-8B-Instruct eu-north1 pricing)."""
    input_cost_per_1k = 0.02 / 1000  # $0.02 per 1M = $0.00002 per 1K
    output_cost_per_1k = 0.06 / 1000  # $0.06 per 1M = $0.00006 per 1K
    
    input_cost = (input_tokens / 1000) * input_cost_per_1k
    output_cost = (output_tokens / 1000) * output_cost_per_1k
    total_cost = input_cost + output_cost
    
    return total_cost


#### Function to generate description per row (per product)

In [10]:
def generate_description(row, config, system_prompt):
    name = row["product_name"]
    attrs = row["Product_attribute_list"]
    material = row["material"]
    warranty = row["warranty"]

    user_prompt = f"""Product name: {name}
        Attributes: {attrs}
        Material: {material}
        Warranty: {warranty}"""

    start_time = time.time()

    response = client.chat.completions.create(
        model=config["model"],
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=config["temperature"],
        top_p=config["top_p"],
        max_tokens=config["max_tokens"],
        extra_body={"top_k": config["top_k"]}
    )

    latency_ms = (time.time() - start_time) * 1000
    description = response.choices[0].message.content.strip()

    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    total_cost = calculate_cost(input_tokens, output_tokens)

    return {
        "generated_description": description,
        "latency_ms": latency_ms,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        'cost_usd': total_cost,
        'length_words': len(description.split())
    }

#### Generate descriptions for all products in our dataset

In [13]:
def process_all_products(
    csv_file='Assignment_01_product_dataset.csv',
    output_file='assignment_01_task_5_03.xlsx',
    config=GEN_CONFIG
):
    """Process entire dataset and save Excel with blank rubric columns."""
    
    df = pd.read_csv(csv_file)
    system_prompt = build_system_prompt()
    
    results = []
    for idx, row in df.iterrows():
        print(f"Processing row {idx+1}/{len(df)}: {row['product_name']}")
        result = generate_description(row, config, system_prompt)
        result['index'] = idx  # Preserve original row index
        results.append(result)
    
    # Create full DataFrame with blank columns
    results_df = pd.DataFrame(results)
    full_df = df.merge(results_df, left_index=True, right_on='index').drop('index', axis=1)

    rubric_cols = ['fluency', 'grammar', 'tone', 'grounding',
                'fluency_explanation', 'grammar_explanation',
                'tone_explanation', 'grounding_explanation', 'length', 'latency', 'cost']
    for col in rubric_cols:
        full_df[col] = ''

    # Map length, latency, and cost to good/ok/bad based on thresholds
    full_df['length'] = full_df['length_words'].apply(lambda x: 'good' if 50 <= x <= 90 else ('ok' if (40 <= x < 50) or (90 < x <= 110) else 'bad'))
    full_df['latency'] = full_df['latency_ms'].apply(lambda x: 'good' if x <= 5000 else ('ok' if 5000 < x <= 9000 else 'bad'))
    full_df['cost'] = full_df['cost_usd'].apply(lambda x: 'good' if x <= 0.0001 else ('ok' if 0.0001 < x <= 0.0005 else 'bad'))
    
    if output_file:
        full_df.to_excel(output_file, index=False)
        print(f"\nSaved {len(full_df)} rows to {output_file}")
    return output_file, full_df

In [14]:
judge_input_file, df_all = process_all_products(output_file=None)

Processing row 1/50: Apple iPhone 15 Pro
Processing row 2/50: Samsung Galaxy S24 Ultra
Processing row 3/50: Google Pixel 8 Pro
Processing row 4/50: Sony WH‑1000XM5 Headphones
Processing row 5/50: Bose QuietComfort Ultra Earbuds
Processing row 6/50: Amazon Echo Dot (5th Gen)
Processing row 7/50: Dell XPS 13 9310 Laptop
Processing row 8/50: Apple MacBook Air 13″ (M3)
Processing row 9/50: Microsoft Surface Pro 10
Processing row 10/50: Garmin Forerunner 255
Processing row 11/50: Fitbit Charge 6
Processing row 12/50: GoPro HERO12 Black
Processing row 13/50: DJI Mini 4 Pro Drone
Processing row 14/50: Nintendo Switch OLED
Processing row 15/50: PlayStation 5 Slim
Processing row 16/50: Xbox Series X
Processing row 17/50: Instant Pot Duo 6‑Quart
Processing row 18/50: Keurig K‑Supreme Plus Smart
Processing row 19/50: Vitamix 5200 Blender
Processing row 20/50: Dyson V15 Detect Vacuum
Processing row 21/50: iRobot Roomba j7+
Processing row 22/50: Yeti Rambler 20 oz Tumbler
Processing row 23/50: Stan

### Task 3 - manual (human) evaluation:

Relevant columns were added to assignment_01.xlsx with pass/fail scores.

Scored 15 products on assignment_01_task_3.xlsx

The best-performing criteria were usually Length, Fluency, and Grammar. The worst-performing criterion was Grounding, because some descriptions included incorrect facts, missing facts, or unsupported details. Since Grounding was part of the no-go rule, it was the main reason some products failed.

### Task 4 - improvement cycle:

Added a config parameter to my workflow, with temperature, top_p, top_k and max_tokens as adjustable parameters

- Experiment 1:
Changed system prompt and asked for less tacky commercial sentences that start with repeating marketing words like “Elevate” and “Discover”...
Reduced temperature to make the sentences less random and more strict and to the point

    Results: Descriptions became highly technical and boring, just dry information with no marketing efforts whatsoever

- Experiment 2:
Increased the temperature (still with the improved prompt)
Removed this part of the prompt:
“Do not start with phrases like 'Discover', 'Experience', 'Introducing', 'Elevate', or similar marketing language” it took all the life out of the description 

    Results: Still highly technical sentences, most starts with “The [product name]..” 

- Experiment 3:
Added “Be friendly, credible, and positive” and “Make it compelling for a customer ready to buy” which transformed the descriptions to be more compelling and commercial like
Other criteria remained as the first experiment, latency was a bit more than the first experiment

    Results: Back to commercial like desriptions but less exaggerated and more concise 

**** Code is the same as the task 2 but i just added the config parameters ****


### Task 5 - create a judge model:

Created a judge model, prompt containing all the relevant rubrics and instructions on how to score them (Fluency, Grammar, Tone, Grounding) all others are calculated programmatically. The judge was asked to be very strict, and should reduce grounding to “ok” on “One minor plausible inference not explicitly stated”.

- Experiment 1: 
    <br>
    <br>
    The judge gave “good” scores to fluency, grammar and tone in all products. A few examples of products where one of the features was something like “battery: long-lasting” made the judge reduce the description score to “ok”. In the explanation it noted that there "plausible inference”

- Experiment 2:
    <br>
    <br>
    Added “specific instructions for handling phrases with “-” which the model did not handle very well
    It improved, still get only “good” for all rubrics, which looks unreliable although the descriptions are fine.

- Experiment 3:
    <br>
    <br>
    Changing model to Qwen3-30B just to see how the results change.
    The larger model did a much better job by identifying some non-relevant features, for example a battery in the feature list of a SanDisk memory card which doesn't have batteries.


In [18]:
JUDGE_CONFIG = {
    "model": "google/gemma-2-9b-it-fast",
    # "model": "Qwen/Qwen3-30B-A3B-Instruct-2507",
    "temperature": 0.1,
    "max_tokens": 500
}

#### Generate pydantic objects

In [19]:
# Pydantic models
class CriterionScore(BaseModel):
    explanation: str = Field(description="Brief reasoning for the verdict")
    verdict: Literal["good", "ok", "bad"] = Field(description="One of: good, ok, bad")

class JudgeOutput(BaseModel):
    fluency: CriterionScore
    grammar: CriterionScore
    tone: CriterionScore
    grounding: CriterionScore

#### Generate general system prompt for the judge 

In [20]:
JUDGE_PROMPT = """You are a strict evaluator of AI-generated e-commerce product descriptions.

Your job is to grade the description using the rubric below. Be conservative and evidence-based.

RUBRIC

Fluency
- good: Reads smoothly and naturally; no awkward phrasing.
- ok: Mostly clear with minor awkward phrasing.
- bad: Choppy or confusing; hard to follow.

Grammar
- good: No noticeable spelling, grammar, or punctuation errors.
- ok: 1-2 minor errors that do not affect understanding.
- bad: Multiple errors that distract or cause confusion.

Tone
- good: Friendly, credible sales voice; positive and persuasive.
- ok: Generally appropriate but slightly too dry, casual, or overhyped.
- bad: Clearly inappropriate, such as negative, rude, sarcastic, robotic, or strongly unnatural.

Grounding
- good: All factual claims match the provided product information; only light generic persuasion is allowed.
- ok: One minor plausible inference not explicitly stated, but not contradictory.
- bad: Invents or contradicts key facts such as material, warranty, or features.

SCORING RULES
- Be strict. Do not give "good" unless the description clearly meets the full definition of good.
- If the description is borderline between two ratings, choose the lower rating.
- Every criterion must include a specific explanation, even when the verdict is "good".
- Explanations must reference concrete wording from the description.
- For grounding, compare factual claims in the description against the product information.
- If unsupported factual details are added, grounding cannot be "good".
- Do not be polite or generous. Your role is accurate grading, not helpful writing.

GROUNDING RULES (be literal):
- Features are listed as "key: value" pairs. "battery: long-lasting" MEANS the battery is long-lasting.
- "energy efficient" MEANS the product is energy efficient.
- If description uses exact phrasing from ANY product field, it is "good".
- Quote the exact matching text in your explanation.

PRODUCT INFORMATION
Product name: {product_name}
Attributes: {attributes}
Material: {material}
Warranty: {warranty}

DESCRIPTION TO EVALUATE
{generated_description}

- Explanations: 1 sentence MAX, 25 words MAX. Quote 1 example.

Return valid JSON that matches the required schema exactly.
"""


#### Function to generate explanations and verdicts per line

In [21]:
def judge_description(row, config=JUDGE_CONFIG):
    prompt = JUDGE_PROMPT.format(
        product_name=row['product_name'],
        attributes=row['Product_attribute_list'],
        material=row['material'],
        warranty=row['warranty'],
        generated_description=row['generated_description']
    )
    
    response = client.chat.completions.create(
        model=config["model"],
        messages=[{"role": "system", "content": prompt}],
        temperature=config["temperature"],
        max_tokens=config["max_tokens"],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "judge_output",
                "schema": JudgeOutput.model_json_schema()
            }
        }
    )
    
    result = JudgeOutput.model_validate_json(response.choices[0].message.content)
    
    # Flatten verdicts for Excel columns
    return {
        "fluency": result.fluency.verdict,
        "fluency_explanation": result.fluency.explanation,
        "grammar": result.grammar.verdict,
        "grammar_explanation": result.grammar.explanation,
        "tone": result.tone.verdict,
        "tone_explanation": result.tone.explanation,
        "grounding": result.grounding.verdict,
        "grounding_explanation": result.grounding.explanation
    }

#### Generate explanations and verdicts for all generated discriptions

In [ ]:
def run_judge(input_file, output_file, row_limit=None):
    """Ensure input and output file paths are provided"""
    if not input_file:        
        raise ValueError("input_file must be provided.")

    """Run judge on all rows with descriptions."""
    print("Loading Excel...")
    df = pd.read_excel(input_file)

    if row_limit:
        df = df.head(row_limit)
        print(f"Limiting to first {row_limit} rows for testing.")

    judge_cols = ['fluency', 'grammar', 'tone', 'grounding',
                    'fluency_explanation', 'grammar_explanation',
                    'tone_explanation', 'grounding_explanation']

    for col in judge_cols:
        if col not in df.columns:
            df[col] = ''
        df[col] = df[col].astype('object')
    
    print(f"Found {len(df)} rows, judging descriptions...")
    
    for idx, row in df.iterrows():
        if pd.notna(row.get('generated_description', '')):
            print(f"Judging row {idx+1}: {row['product_name']}...")
            try:
                scores = judge_description(row)
                df.loc[idx, 'fluency'] = scores['fluency']
                df.loc[idx, 'grammar'] = scores['grammar']
                df.loc[idx, 'tone'] = scores['tone']
                df.loc[idx, 'grounding'] = scores['grounding']
                df.loc[idx, 'fluency_explanation'] = scores['fluency_explanation']
                df.loc[idx, 'grammar_explanation'] = scores['grammar_explanation']
                df.loc[idx, 'tone_explanation'] = scores['tone_explanation']
                df.loc[idx, 'grounding_explanation'] = scores['grounding_explanation']
            except Exception as e:
                print(f"Error judging row {idx}: {e}")
                df.loc[idx, 'fluency'] = 'error'
    if output_file:
        output_file = output_file.replace('.xlsx', '_judged.xlsx')
        df.to_excel(output_file, index=False)
        print(f"Saved judged results to {output_file}")
    return df

#### Functions to generate final score for each row

In [ ]:
def pass_or_fail(row):
    """Calculate final score based on rubric criteria.
    No-go rules: 
        grounding ≠ good
        length = bad
        tone = bad

    Let G = # good, O = # ok, B = # bad
    Pass: G ≥ 3 ∧ O ≤ 2 ∧ B = 0
    """
    criteria = ['fluency', 'grammar', 'tone', 'grounding', 'length', 'latency', 'cost']
    verdicts = [row[crit] for crit in criteria]
    if row['grounding'] != 'good' or row['length'] == 'bad' or row['tone'] == 'bad':
        return 'fail'
    good_count = verdicts.count('good')
    ok_count = verdicts.count('ok')
    bad_count = verdicts.count('bad')
    if good_count >= 3 and ok_count <= 2 and bad_count == 0:
        return 'pass'
    else:
        return 'fail'   
    
def generate_final_score(df, final_output_file=None):
    """Apply pass/fail logic to entire DataFrame."""
    df['final_score'] = df.apply(pass_or_fail, axis=1)
    if final_output_file:
        final_output_file = final_output_file.replace('.xlsx', '_final.xlsx')
        df.to_excel(final_output_file, index=False)
        print(f"Saved final scores to {final_output_file}")
    return df

### Task 6 - run and analyze the judge:

In [ ]:
df_judged_limit_5 = run_judge(input_file='assignment_01_task_6_03.xlsx', output_file=None, row_limit=5)
df_judged_limit_5_final = generate_final_score(df_judged_limit_5)

Loading Excel...
Limiting to first 5 rows for testing.
Found 5 rows, judging descriptions...
Judging row 1: Apple iPhone 15 Pro...
Judging row 2: Samsung Galaxy S24 Ultra...
Judging row 3: Google Pixel 8 Pro...
Judging row 4: Sony WH‑1000XM5 Headphones...
Judging row 5: Bose QuietComfort Ultra Earbuds...
Saved judged results to None


#### Sanity check:
    Agreed on all products moved to full run

In [ ]:
df_judged = run_judge(input_file='assignment_01_task_6_03.xlsx', output_file=None)
df_judged_final = generate_final_score(df_judged)

Loading Excel...
Found 50 rows, judging descriptions...
Judging row 1: Apple iPhone 15 Pro...
Judging row 2: Samsung Galaxy S24 Ultra...
Judging row 3: Google Pixel 8 Pro...
Judging row 4: Sony WH‑1000XM5 Headphones...
Judging row 5: Bose QuietComfort Ultra Earbuds...
Judging row 6: Amazon Echo Dot (5th Gen)...
Judging row 7: Dell XPS 13 9310 Laptop...
Judging row 8: Apple MacBook Air 13″ (M3)...
Judging row 9: Microsoft Surface Pro 10...
Judging row 10: Garmin Forerunner 255...
Judging row 11: Fitbit Charge 6...
Judging row 12: GoPro HERO12 Black...
Judging row 13: DJI Mini 4 Pro Drone...
Judging row 14: Nintendo Switch OLED...
Judging row 15: PlayStation 5 Slim...
Judging row 16: Xbox Series X...
Judging row 17: Instant Pot Duo 6‑Quart...
Judging row 18: Keurig K‑Supreme Plus Smart...
Judging row 19: Vitamix 5200 Blender...
Judging row 20: Dyson V15 Detect Vacuum...
Judging row 21: iRobot Roomba j7+...
Judging row 22: Yeti Rambler 20 oz Tumbler...
Judging row 23: Stanley Quencher H2.

#### Full run:
As described in the last section, ran the judge on all products, changed the prompt, and changed the model. 

The bigger model was more strict and gave ‘ok’ and ‘bad’ ratings on fluency, grammar and tone, however i either did not agree with his decision or he made false claims about some phrases in the generated description
Going back to to the small judge model (gemma) it gave ‘good’ rating for all and some ‘ok’ for tone but it did not fail any products.



#### Criterion-by-criterion judging:

In [29]:
class SingleCriterionScore(BaseModel):
    explanation: str
    verdict: Literal["good", "ok", "bad"]

#### Generate general system prompt for the criterion-by-criterion judge 

In [30]:
CRITERION_PROMPT = """You are a strict evaluator of AI-generated e-commerce product descriptions.

Your job is to grade the description using the rubric below. Be conservative and evidence-based.

CRITERION: {criterion_name}
{criterion_definition}

SCORING RULES
- Be strict. Do not give "good" unless the description clearly meets the full definition of good.
- If the description is borderline between two ratings, choose the lower rating.
- Every criterion must include a specific explanation, even when the verdict is "good".
- Explanations must reference concrete wording from the description.
- Do not be polite or generous. Your role is accurate grading, not helpful writing.

PRODUCT INFORMATION
Product name: {product_name}
Attributes: {attributes}
Material: {material}
Warranty: {warranty}

DESCRIPTION TO EVALUATE
{generated_description}

- Explanations: MAXIMUM 25 WORDS, NO MORE."

Return valid JSON: {{"explanation": "...", "verdict": "good|ok|bad"}}
"""

# Specific definitions
CRITERIA = {
    "fluency": "good: Reads smoothly and naturally; no awkward phrasing. ok: Mostly clear with minor awkward phrasing. bad: Choppy or confusing; hard to follow. FLUENCY RULES: Ignore whether claims are factual. Only judge sentence flow and readability.",
    "grammar": "good: No noticeable spelling, grammar, or punctuation errors. ok: 1-2 minor errors that do not affect understanding. bad: Multiple errors that distract or cause confusion. GRAMMER RULES: Ignore whether claims are factual. Only judge grammar, punctuation, and spelling.",
    "tone": "good: Friendly, credible sales voice; positive and persuasive. ok: Generally appropriate but slightly too dry, casual, or overhyped. bad: Clearly inappropriate, such as negative, rude, sarcastic, robotic, or strongly unnatural. TONE RULES: Ignore whether claims are factual. Only judge the style and attitude of the writing.",
    "grounding": "good: All factual claims match the provided product information; only light generic persuasion is allowed. ok: One minor plausible inference not explicitly stated, but not contradictory. bad: Invents or contradicts key facts such as material, warranty, or features. GROUNDING RULES (be literal): Features listed as 'key: value' pairs are explicit facts. 'battery: long-lasting' = battery has long-lasting feature. 'energy efficient' MEANS the product is energy efficient. Paraphrasing provided features = grounded. Quote the exact matching text in your explanation."
}

#### Function to generate explanations and verdicts per line AND per criterion (each row runs 4 times)

In [34]:
def judge_single_criterion(row, criterion):
    prompt = CRITERION_PROMPT.format(
        criterion_name=criterion,
        criterion_definition=CRITERIA[criterion],
        product_name=row['product_name'],
        attributes=row['Product_attribute_list'],
        material=row['material'],
        warranty=row['warranty'],
        generated_description=row['generated_description']
    )
    
    response = client.chat.completions.create(
        model=JUDGE_CONFIG["model"],
        messages=[{"role": "system", "content": prompt}],
        temperature=0.0,
        max_tokens=JUDGE_CONFIG["max_tokens"],  # Much shorter now
        response_format={"type": "json_object"}  # Simpler schema
    )
    
    result = SingleCriterionScore.model_validate_json(response.choices[0].message.content)
    return {
            "verdict": result.verdict,
            "explanation": result.explanation
        }

#### Generate explanations and verdicts for all generated discriptions with criterion-by-criterion judge

In [ ]:
def run_judge_isolated(input_file, output_file):
    df = pd.read_excel(input_file)

    criteria = ['fluency', 'grammar', 'tone', 'grounding']

    for criterion in criteria:
        verdict_col = f"{criterion}"
        explanation_col = f"{criterion}_explanation"

        if verdict_col not in df.columns:
            df[verdict_col] = ''
        if explanation_col not in df.columns:
            df[explanation_col] = ''

        df[verdict_col] = df[verdict_col].astype('object')
        df[explanation_col] = df[explanation_col].astype('object')

    print(f"Found {len(df)} rows, judging isolated criteria...")

    for idx, row in df.iterrows():
        if pd.notna(row.get('generated_description', '')):
            print(f"Judging row {idx+1}: {row['product_name']}")

            for criterion in criteria:
                try:
                    result = judge_single_criterion(row, criterion)

                    df.loc[idx, f"{criterion}"] = result["verdict"]
                    df.loc[idx, f"{criterion}_explanation"] = result["explanation"]

                except Exception as e:
                    print(f"Error on row {idx}, criterion {criterion}: {e}")
                    df.loc[idx, f"{criterion}"] = 'error'
                    df.loc[idx, f"{criterion}_explanation"] = str(e)

    if output_file:
        output_file = output_file.replace('.xlsx', '_judged_isolated.xlsx')
        df.to_excel(output_file, index=False)
        print(f"Saved isolated judged results to {output_file}")
    return df

In [ ]:
df_judged_isolated = run_judge_isolated(input_file='assignment_01_task_6_03.xlsx', output_file=None)
df_judged_isolated_final = generate_final_score(df_judged_isolated)

Found 50 rows, judging isolated criteria...
Judging row 1: Apple iPhone 15 Pro
Judging row 2: Samsung Galaxy S24 Ultra
Judging row 3: Google Pixel 8 Pro
Judging row 4: Sony WH‑1000XM5 Headphones
Judging row 5: Bose QuietComfort Ultra Earbuds
Judging row 6: Amazon Echo Dot (5th Gen)
Judging row 7: Dell XPS 13 9310 Laptop
Judging row 8: Apple MacBook Air 13″ (M3)
Judging row 9: Microsoft Surface Pro 10
Judging row 10: Garmin Forerunner 255
Judging row 11: Fitbit Charge 6
Judging row 12: GoPro HERO12 Black
Judging row 13: DJI Mini 4 Pro Drone
Judging row 14: Nintendo Switch OLED
Judging row 15: PlayStation 5 Slim
Judging row 16: Xbox Series X
Judging row 17: Instant Pot Duo 6‑Quart
Judging row 18: Keurig K‑Supreme Plus Smart
Judging row 19: Vitamix 5200 Blender
Judging row 20: Dyson V15 Detect Vacuum
Judging row 21: iRobot Roomba j7+
Judging row 22: Yeti Rambler 20 oz Tumbler
Judging row 23: Stanley Quencher H2.0 40 oz
Judging row 24: Hydro Flask 32 oz Wide Mouth
Judging row 25: Contigo A

- Does this change the results? 
    <br>
    <br>
    It did not change the results for most products, however when using the criterion-by-criterion method the explanations are easier to understand and agree with. The normal method did not score 'fluency', 'grammer' and 'tone' anything other than 'good' which i found unreliable.
    <br>
    
- Why might isolating criteria lead to different outcomes? 
    <br>
    <br>
    By isolating the criteria we reduce the cognitive load of the LLM judge and sharpen his focus on each selected criteria. The prompt is the isolated version is shorter than the general version, it doesn't hold all 4 rubics in the working memory just one. The judge also manages less context which reduces the chances for confusion
    <br>

#### Analysis:
1. What are the practical trade-offs between human evaluation and LLM-as-a-judge (think: cost, scale, consistency, accuracy)?
<br>
    When considering cost and scale using LLM's will have the advantage of being highly cost effective compared to a human evaluator. LLM will also be highly consistent, since it is following exactly the same instructions for each product, so as long as you are happy with it's performance consistency is another advantage for the LLM.
    accuracy is the only aspect where the LLM can go wrong if not ensuring it does the work as intended and a human evaluator will know exactly if it approves or disapproves of the generated text.

2. Which approach would you recommend for a production system that generates thousands of descriptions daily?
<br>
    Production scale system should work with LLM judges and have sampled random inspections of their performance. It will not be possible for a human to evaluate thousands of descriptions daily, nor it will be effective in any way. 